# Train one LoRA adapter on Colab

Runtime → Change runtime type → **T4 GPU**. Put `train_lora.py` and `questions.csv` in `MyDrive/lora_runs/`
(once), fill in the parameters, then Runtime → Run all.

One run = one adapter for one subject. The adapter (~70 MB) is pushed to a private Hugging Face repo;
the base model is downloaded here and discarded with the session. Training energy is logged with CodeCarbon
and saved inside the adapter repo as `training_summary.json`. Checkpoints go to Drive under `lora_runs/<repo>/`.

In [ ]:
#@title Parameters
BASE = "Qwen/Qwen2.5-3B-Instruct"   #@param ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]
SUBJECT = "math"                   #@param ["math", "code", "knowledge"]
HF_USER = "your-hf-username"       #@param {type:"string"}
N_EXAMPLES = 2000                  #@param {type:"integer"}
EPOCHS = 2                         #@param {type:"number"}
RANK = 16                          #@param {type:"integer"}
# FOUR_BIT: keep False for 1.5B/3B so the adapter matches the fp16 base the Mac serves; True for the 7B (QLoRA)
FOUR_BIT = False                   #@param {type:"boolean"}
# COUNTRY: ISO-3 code for the training-energy CO2 factor
COUNTRY = "IND"                    #@param {type:"string"}

assert HF_USER != "your-hf-username", "set HF_USER to your Hugging Face username"
REPO = f"{HF_USER}/{BASE.split('/')[-1].lower()}-lora-{SUBJECT}"
print("adapter will be pushed to:", REPO)

In [ ]:
#@title Install (pinned so the adapter loads on the Mac with the same peft major version)
!pip -q install "transformers>=4.45,<5" "peft>=0.13" "datasets>=2.20" "accelerate" "codecarbon>=2.8" "huggingface_hub" bitsandbytes
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
#@title Log in to Hugging Face (WRITE token; accept the base model's licence first if it is gated)
# Preferred for Run-all: store the token as a Colab secret named HF_TOKEN (key icon, left sidebar).
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("logged in from the HF_TOKEN secret")
except Exception:
    login()   # interactive prompt

In [ ]:
#@title Mount Drive (script, evaluation bank and checkpoints live there)
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/lora_runs"   #@param {type:"string"}
OUT = os.path.join(DRIVE_DIR, REPO.split("/")[-1])
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)

In [ ]:
#@title Copy train_lora.py and questions.csv from Drive into the session
# Keep both files in DRIVE_DIR (set above). questions.csv is optional: if present, any training row
# whose question is in the evaluation bank is dropped. Fallback: upload them via the Files panel.
import os, shutil
for f in ("train_lora.py", "questions.csv"):
    src = os.path.join(DRIVE_DIR, f)
    if os.path.exists(src) and not os.path.exists(f):
        shutil.copy(src, f)
assert os.path.exists("train_lora.py"), f"train_lora.py not found in {DRIVE_DIR} or the session"
print("train_lora.py ready;", "questions.csv ready" if os.path.exists("questions.csv") else "questions.csv missing (exclusion filter skipped)")

In [ ]:
#@title Train (3B on T4: ~1-2 h for 2,000 examples x 2 epochs; 7B QLoRA: ~3-4 h)
# Built as a Python list and run with subprocess: Colab's $var substitution and backslash
# continuations in a ! line mangled the argument list.
import os, subprocess, sys, shlex
cmd = [sys.executable, "-u", "train_lora.py",
       "--base", BASE, "--subject", SUBJECT, "--n", str(N_EXAMPLES), "--epochs", str(EPOCHS),
       "--rank", str(RANK), "--out", OUT, "--push-to", REPO, "--country", COUNTRY]
if FOUR_BIT:
    cmd.append("--four-bit")
if os.path.exists("questions.csv"):      # evaluation bank: matching training rows are dropped
    cmd += ["--exclude", "questions.csv"]
print(" ".join(shlex.quote(c) for c in cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
#@title Training summary (also saved inside the adapter repo)
import json
print(json.dumps(json.load(open(f"{OUT}/adapter/training_summary.json")), indent=2))
print("adapter repo: https://huggingface.co/" + REPO)